Raw Memory, Strides & C-Types (The Direct Triton/CUDA Bridge)

Triton, CUDA, and C++ kernels do not work with high-level Python objects. They operate directly on raw memory pointers, byte offsets, and stride arithmetic.

1. Strides and The General Offset FormulaWhen a multi-dimensional tensor is flattened into 1D memory, strides define how many elements (or bytes) you must jump in physical memory to move 1 step along each dimension.Row-Major (C-Style, Default in Python/PyTorch/C++): Last dimension is contiguous (stride = 1).Formula for any N-D index $(i_0, i_1, \dots, i_{n-1})$:$$\text{Memory Offset} = \sum_{k=0}^{n-1} (i_k \times \text{stride}_k)$$For a 3D Tensor of shape $(D, H, W)$:$$\text{Offset}(d, h, w) = d \times (H \times W) + h \times W + w \times 1$$In Triton kernels, you write this stride math explicitly to load block pointers from global memory:tl.load(ptr + offsets_row[:, None] * stride_r + offsets_col[None, :] * stride_c).

1. Strides = “how far do I jump?”

Suppose:

A B C
D E F

Memory is actually:

[A, B, C, D, E, F]

Each element has a position:

A=0  B=1  C=2  D=3  E=4  F=5

A stride tells you:

“If I move 1 step in this dimension, how many memory elements do I jump?”

For this 2×3 array:

shape   = (2, 3)
strides = (3, 1)

Why?

Move one row down → jump 3 elements → stride_row = 3
Move one column right → jump 1 element → stride_col = 1

So:

offset = row * stride_row + col * stride_col

For E:

offset = 1 * 3 + 1 * 1
       = 4

So E lives at buffer[4].

This is the important mental model

A tensor index:

tensor[r, c]

is really just:

base_pointer + r*stride_r + c*stride_c

That's basically what you'll write in Triton/CUDA.

2. Why transpose doesn't need copying

Normally you might imagine transpose doing:

[A B C]       [A D]
[D E F]  →    [B E]
              [C F]

But you don't actually need to move anything.

Original:

shape   = (2, 3)
strides = (3, 1)

Transposed view:

shape   = (3, 2)
strides = (1, 3)

Same memory!

[A B C D E F]

Only the rules for calculating offsets changed.

So:

transposed[1, 0]

means:

1 * 1 + 0 * 3 = 1

→ B

This is called a view.

View = different way of looking at the same memory.

That's why modifying the transpose also modifies the original.

3. memoryview = Python's window into raw memory

Normally Python hides memory details from you.

x = [1, 2, 3]

You don't care where 1, 2, 3 physically live.

memoryview gets you closer to the hardware:

raw = bytearray(...)
mv = memoryview(raw)

Now you're saying:

“Give me access to the underlying buffer without copying it.”

And:

mv.cast("i")

means roughly:

“Interpret every 4 bytes as a C-style 32-bit integer.”

So if bytes represent:

01 00 00 00 | 02 00 00 00

you can see them as:

[1, 2]
Important distinction

memoryview doesn't magically convert the data.

It reinterprets the same bytes.

That's very important when working with low-level systems.

4. ctypes = Python ↔ C bridge

Now we're one level lower.

import ctypes

ctypes lets Python talk directly to native C libraries.

For example:

c_int
c_float
c_double
c_void_p

roughly correspond to:

C              Python ctypes

int       →    c_int
float     →    c_float
double    →    c_double
void*     →    c_void_p

So:

ctypes.c_float

means:

“I want a value that looks like a C float.”

5. ctypes pointers

This:

ctypes.POINTER(ctypes.c_float)

means:

pointer → float

Think:

0x7FFA1234
      │
      ▼
   float

And:

ctypes.addressof(c_array)

gives you the actual memory address.

For example:

0x7f83a21c4000

That's the kind of thing CUDA/Triton ultimately operates around.

In [1]:
raw_bytes = bytearray(b"\x01\x00\x00\x00\x02\x00\x00\x00") # 8 bytes
mv = memoryview(raw_bytes).cast("i") # Reinterpret as 32-bit signed integers (4 bytes each)

print(mv[0])  # 1
print(mv[1])  # 2

# Mutating mv directly mutates the raw backing bytearray!
mv[0] = 99
print(raw_bytes)  # bytearray contains the updated raw bytes

1
2
bytearray(b'c\x00\x00\x00\x02\x00\x00\x00')


Think of memory as boxes

Imagine RAM is a long row of 1-byte boxes:

RAM:

[ box ][ box ][ box ][ box ][ box ][ box ][ box ][ box ]
   1     2     3     4     5     6     7     8

Each box = 1 byte = 8 bits.

Now suppose I want to store a 32-bit int.

A 32-bit int needs:

32 bits = 4 bytes

So it occupies 4 boxes:

[       32-bit integer       ][       another       ]
[ 1 byte ][ 1 ][ 1 ][ 1 ]    [ 1 ][ 1 ][ 1 ][ 1 ]
Now here's what cast("i") does

The memory itself doesn't change.

Before casting, Python looks at it as:

[byte][byte][byte][byte][byte][byte][byte][byte]

After:

memoryview(...).cast("i")

you're telling Python:

"Hey, treat every 4 bytes together as one integer."

So Python now sees:

[int  ][int  ]
 4B      4B

Nothing was converted from 32-bit to 48-bit or anything like that.

We're simply changing how we interpret the same memory.

Super concrete example

Suppose memory contains:

01 00 00 00

That's 4 bytes.

If we interpret those 4 bytes as a 32-bit integer, we get:

1

Now imagine:

01 00 00 00 | 02 00 00 00

Raw-byte interpretation:

[01][00][00][00][02][00][00][00]

cast("i") interpretation:

[     1     ][     2     ]
   4 bytes      4 bytes
Why do we want this?

Because our tensor might be:

[1, 2]

where each number is a 32-bit int.

We don't want to manually deal with:

byte 0
byte 1
byte 2
byte 3
...

We want:

buf[0]  # 1
buf[1]  # 2

So cast tells Python what a "single element" means.

One sentence to remember

Memory is just bytes. cast() tells Python how many bytes should be grouped together and what type they represent.

So:

"b" → 1 byte per element
"i" → 4 bytes per element
"f" → 4 bytes per element
"d" → 8 bytes per element

And 32-bit = 4 bytes, not 48 bits.

That's the whole idea.

Exercise 1: Zero-Copy Strided Tensor Slice Engine
Build a pure Python class StridedArray2D without using NumPy:

Requirements:

__init__(self, raw_buffer: bytearray, shape: tuple[int, int], strides: tuple[int, int] = None):

Store shape = (rows, cols).

If strides is not provided, compute default row-major element strides: strides = (cols, 1).

Store self.buf = memoryview(raw_buffer).cast("i") (treating every 4 bytes as a 32-bit int).

__getitem__(self, key: tuple[int, int]):

Unpack r, c = key.

Compute flat offset using stride formula: offset = r * self.strides[0] + c * self.strides[1].

Return self.buf[offset].

transpose(self):

Returns a new StridedArray2D pointing to the exact same memory buffer (self.buf), but with:

shape = (cols, rows)

strides = (self.strides[1], self.strides[0]) (swapped strides!).

Verify: Modifying an element in the transposed view instantly reflects in the original array because no data was copied!

In [3]:
class StrideArray2D:
    def __init__(
        self,
        raw_buffer: bytearray,
        shape: tuple[int, int],
        strides: tuple[int, int] = None
    ):
        self.row, self.col = shape

        if strides is None:
            self.strides = (self.col, 1)
        else:
            self.strides = strides

        self.buf = memoryview(raw_buffer).cast("i")
        self.raw_buffer = raw_buffer

    def __getitem__(self, key: tuple[int, int]):
        r, c = key

        r_stride, c_stride = self.strides

        offset = r * r_stride + c * c_stride

        return self.buf[offset]

    def __setitem__(self, key: tuple[int, int], value: int):
        r, c = key

        r_stride, c_stride = self.strides

        offset = r * r_stride + c * c_stride

        self.buf[offset] = value

    def transpose(self):
        return StrideArray2D(
            self.raw_buffer,
            shape=(self.col, self.row),
            strides=(self.strides[1], self.strides[0])
        )

In [4]:
# Create raw memory for 6 int32 values
raw = bytearray(24)

# Put values into the buffer
buf = memoryview(raw).cast("i")

values = [1, 2, 3, 4, 5, 6]

for i, value in enumerate(values):
    buf[i] = value


# Create our 2D array
arr = StrideArray2D(
    raw,
    shape=(2, 3)
)

print("Original:")
print(arr[0, 0], arr[0, 1], arr[0, 2])
print(arr[1, 0], arr[1, 1], arr[1, 2])


# Transpose
t = arr.transpose()

print("\nTransposed:")
print(t[0, 0], t[0, 1])
print(t[1, 0], t[1, 1])
print(t[2, 0], t[2, 1])


# Modify transpose
t[0, 1] = 99

print("\nAfter t[0,1] = 99:")

print("Transpose:")
print(t[0, 0], t[0, 1])
print(t[1, 0], t[1, 1])
print(t[2, 0], t[2, 1])

print("\nOriginal:")
print(arr[0, 0], arr[0, 1], arr[0, 2])
print(arr[1, 0], arr[1, 1], arr[1, 2])

Original:
1 2 3
4 5 6

Transposed:
1 4
2 5
3 6

After t[0,1] = 99:
Transpose:
1 99
2 5
3 6

Original:
1 2 3
99 5 6
